# Competição · 1/4 · modelos de CPU (10 modelos, 10 temporadas)

**Antes de rodar (uma vez só):** em `MyDrive/dengue_V4/` (a pasta do V4, que já tem `dados/` e `referencias_imdc3/`) suba `comp_lib.py`, `comp_modelos.py` e `comp_inla.R`, e **substitua o `v4_lib.py`** pela versão nova de `data_imdc_2026/V4/`.
Todos os resultados vão para `MyDrive/dengue_V4/resultados_competicao/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.

| Notebook | Modelos | Ambiente |
|---|---|---|
| `Comp1_CPU` | climatologia, canal_endemico, AR_longo, SURGE, ARIMA_Fourier, ETS_MSTL, Prophet, LightGBM_direto, Renovacao_S, Tamanho_x_forma | CPU |
| `Comp2_INLA` | BB_INLA, GHR_INLA | CPU com bastante RAM |
| `Comp3_GPU` | XGBoost_quantis, DeepAR, NHITS, TFT, TSMixer, Chronos, AGCRN | GPU L4 |
| `Comp4_Avaliacao` | rankings, gráficos, diagnóstico, ensembles | qualquer (CPU) |

Os notebooks 1, 2 e 3 podem rodar **ao mesmo tempo**, cada um numa sessão. O 4 pode rodar a qualquer momento: ele avalia o que já existir.

**Ambiente:** CPU (não gasta GPU). **Tempo estimado:** 1–1,5 h. Nos testes do PC, por temporada: climatologia e canal ~1 s, AR_longo ~10 s, SURGE ~3 s, ETS ~12 s, Prophet ~40 s, LightGBM ~1,3 min, ARIMA ~2 min, Renovação ~5 s, Tamanho×forma ~8 s.

In [ ]:
NOTEBOOK = 'cpu'
PACOTES = 'statsforecast'

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/comp', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'dados', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM
for m in (V, L, CM):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = L.Contexto(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'))
TODAS = list(range(2016, 2026))          # 2016–2021: calibração dos pesos do ensemble · 2022–2025: S1–S4
RECENTES = list(range(2020, 2026))       # para os modelos mais caros
OFICIAIS = V.TEMPORADAS_OFICIAIS
M = 500                                  # trajetórias por região
REFAZER = False

In [ ]:
MODELOS = {m.nome: m for m in [CM.Climatologia(), CM.CanalEndemico(), CM.ARLongo(), CM.Surge(), CM.ARIMAFourier(),
                               CM.ETSMSTL(), CM.ProphetModelo(), CM.LightGBMDireto(), CM.RenovacaoS(), CM.TamanhoForma()]}
PLANO = {nome: TODAS for nome in MODELOS}

In [ ]:
# @title Rodar (retoma de onde parou; um modelo com erro não interrompe os outros)
erros = []
for nome, temps in PLANO.items():
    for s in temps:
        try:
            dt = L.rodar(ctx, MODELOS[nome], s, PASTA_RES, M=M, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:18s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)))
            print(f'❌ {nome:18s} {s}: {repr(e)[:400]}', flush=True)
            traceback.print_exc(limit=2)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_RES, f'erros_{NOTEBOOK}.csv'), index=False)
t = L.tempos(PASTA_RES)
t[t.modelo.isin(list(PLANO))].groupby('modelo').segundos.agg(['count', 'mean', 'sum']).div([1, 60, 60]).round(1) \
    .rename(columns={'count': 'temporadas', 'mean': 'min/temporada', 'sum': 'min total'})